### 10. Model Evaluation and Optimization

**Objective:** Evaluate and compare the three recommendation models built in Tasks 6–9:
1. **Content-Based (CB)** - TF-IDF + cosine on Title+Description+Tags (Task 6 Exp3), queried with a
   **multi-talk user profile** for evaluation.
2. **Collaborative Filtering (CF)** - **mean-centered** user-based CF on simulated ratings (Task 8).
3. **Hybrid** - min-max fusion of CB and confidence-weighted CF (Task 9).

**Data:** 500 users x 200 talks, ~9,800 ratings (~90% sparse).

**Metrics:** **Precision@K**, **Recall@K**, and **NDCG@K** on a held-out set of each user's ratings.
- **Relevant** = user rated the talk **≥ 3**.
- **K = 5** (project standard) and **K = 10** (robustness check).
- **α grid** for hybrid: `{0.0, 0.3, 0.5, 0.7, 1.0}`.

**Evaluation protocol:**
- For each user: hold out **30% of their ratings** (random, seeded).
- CB query: average the TF-IDF vectors of the user's training talks rated **≥ 4** (fallback: top-rated
  train talk), then score the 200-talk candidate universe against that profile.
- CF: per-user means and mean-centered similarity are computed on the **training split only**.
- Generate top-K recommendations; a recommendation is a **hit** if its `talk_id` is in the held-out
  relevant set.
- Precision@K = hits / K; Recall@K = hits / (relevant items held out); NDCG@K = DCG@K / IDCG@K.

**Matched candidate universe:** CB ranks only the **200 talk_ids that appear in `user_interactions.csv`**,
and CF ranks only talks present in the training interaction matrix (the same 200 talks). This removes
the candidate-universe advantage that previously made CB look ~25x worse than it is. An appendix
reports the unrestricted CB result for comparison only.

In [57]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 180)

PROCESSED_DIR  = Path("../data/processed")
INTERACTION_DIR = Path("../data/interaction")

##### Load Data and Artefacts

Loads CB artefacts (Task 6), the mean-centered CF model (Task 8), and the interactions dataset.
The CB TF-IDF matrix is needed to build per-user profiles; the CF model supplies the neighbour
count `k`. Train-time CF statistics are recomputed below on the training split only.

In [58]:
df_talks = pd.read_csv(PROCESSED_DIR / "ted_talks_cleaned.csv").reset_index(drop=True)
interactions = pd.read_csv(INTERACTION_DIR / "user_interactions.csv")

print(f"Talks:        {df_talks.shape[0]}")
print(f"Interactions: {interactions.shape[0]}")
print(f"Users:        {interactions['user_id'].nunique()}")
print(f"Interacted talks (candidate universe): {interactions['talk_id'].nunique()}")

Talks:        4005
Interactions: 9778
Users:        500
Interacted talks (candidate universe): 200


In [59]:
# ---- CB artefacts ----
CB_TFIDF_PATH = PROCESSED_DIR / "tfidf_matrix_exp3.npz"
CB_COSIM_PATH = PROCESSED_DIR / "cosine_sim_exp3.npy"
CB_VEC_PATH   = PROCESSED_DIR / "tfidf_vectorizer_exp3.joblib"

if CB_TFIDF_PATH.exists() and CB_COSIM_PATH.exists():
    cb_tfidf = load_npz(CB_TFIDF_PATH)
    cb_cosim = np.load(CB_COSIM_PATH)
    cb_vec   = joblib.load(CB_VEC_PATH)
    print("CB artefacts loaded.")
else:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from scipy.sparse import save_npz
    print("Rebuilding CB artefacts (Exp3)...")
    cb_vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_df=0.9)
    combined = (
        df_talks["title"].fillna("") + " " +
        df_talks["description"].fillna("") + " " +
        df_talks["tags"].fillna("").str.replace(",", " ", regex=False)
    )
    cb_tfidf = cb_vec.fit_transform(combined)
    cb_cosim = cosine_similarity(cb_tfidf, dense_output=True)
    np.fill_diagonal(cb_cosim, 0.0)
    save_npz(CB_TFIDF_PATH, cb_tfidf); np.save(CB_COSIM_PATH, cb_cosim); joblib.dump(cb_vec, CB_VEC_PATH)
    print("Saved.")

# ---- CF model (Task 8) ----
CF_MODEL_PATH = PROCESSED_DIR / "cf_model.joblib"
if CF_MODEL_PATH.exists():
    cf_model = joblib.load(CF_MODEL_PATH)
    K_NEIGHBORS = int(cf_model.get("k", 20))
    print("CF model loaded.")
else:
    K_NEIGHBORS = 20
    print("CF model missing - using k = 20; train-time CF is rebuilt below.")

print(f"CB TF-IDF shape:  {cb_tfidf.shape}")
print(f"CB cosine shape:  {cb_cosim.shape}")
print(f"CF neighbours k:  {K_NEIGHBORS}")

CB artefacts loaded.
CF model loaded.
CB TF-IDF shape:  (4005, 23643)
CB cosine shape:  (4005, 4005)
CF neighbours k:  20


##### Train / Test Split (Leave-out 30% per User)

For each user, **hide 30% of their ratings** (at least 1 item). Only the hidden talks that are **≥ 3**
count as relevant.

A user is evaluable when they have at least one relevant held-out talk. Users with no relevant
held-out item are skipped because recall is undefined.

The **training user-item matrix** is used to compute per-user means and similarity for CF. All
statistics are computed on `train_df` only, so there is no leakage.

In [60]:
def train_test_split_per_user(interactions, holdout_frac=0.3, seed=42):
    train_rows, test_rows = [], []
    for uid, grp in interactions.groupby("user_id"):
        grp = grp.sample(frac=1, random_state=seed).reset_index(drop=True)
        n_test = max(1, int(len(grp) * holdout_frac))
        test_rows.append(grp.iloc[:n_test])
        train_rows.append(grp.iloc[n_test:])
    return pd.concat(train_rows, ignore_index=True), pd.concat(test_rows, ignore_index=True)

train_df, test_df = train_test_split_per_user(interactions, holdout_frac=0.3, seed=42)

print(f"Train interactions: {len(train_df)}")
print(f"Test  interactions: {len(test_df)}")

# Build the TRAINING user-item matrix
train_user_item = train_df.pivot_table(
    index="user_id", columns="talk_id", values="rating", aggfunc="mean"
)

# Per-user means and mean-centered matrix, computed on TRAIN only (no leakage)
train_user_means = train_user_item.mean(axis=1)
train_centered = train_user_item.sub(train_user_means, axis=0).fillna(0.0)

# Recompute user-user similarity on the centered TRAIN matrix
train_sim = cosine_similarity(train_centered.values, dense_output=True)
np.fill_diagonal(train_sim, 0.0)
train_sim_df = pd.DataFrame(train_sim, index=train_user_item.index, columns=train_user_item.index)

print(f"Train user-item shape: {train_user_item.shape}")

Train interactions: 7059
Test  interactions: 2719
Train user-item shape: (500, 200)


In [61]:
# Relevant held-out talks (rating >= 3) per user
RELEVANCE_THRESHOLD = 3
N_USERS = interactions["user_id"].nunique()

relevant_heldout = (
    test_df[test_df["rating"] >= RELEVANCE_THRESHOLD]
    .groupby("user_id")["talk_id"]
    .apply(set)
    .to_dict()
)

# Evaluate users with at least 1 relevant held-out talk
evaluable_users = [uid for uid, r in relevant_heldout.items() if len(r) > 0]
print(f"Evaluable users: {len(evaluable_users)} / {N_USERS}")
print(f"Avg relevant held-out per evaluable user: {np.mean([len(relevant_heldout[u]) for u in evaluable_users]):.2f}")

Evaluable users: 489 / 500
Avg relevant held-out per evaluable user: 3.30


##### Models

Each model returns a list of `talk_id`s (top-K), excluding talks the user already rated in **train**.
This mirrors how the models would behave in production.

- **CB:** builds a per-user TF-IDF profile from training talks rated **≥ 4** (fallback: top-rated
  train talk) and scores the **200 interacted talks** against it.
- **CF:** mean-centered user-based CF on the **train** matrix; predictions are clipped to [1, 5] and
  carry `n_raters` for confidence weighting.
- **Hybrid:** min-max fuses CB and confidence-weighted CF.

##### CB Query Strategy for Evaluation (model unchanged)

> **Important:** the content-based **model** is exactly the one built in Tasks 5/6 - the same
> TF-IDF matrix (`tfidf_matrix_exp3.npz`) and the same cosine geometry. Nothing about the model is
> retrained or altered here.
>
> Only the **query strategy used for evaluation** changes: instead of a single seed talk, we build a
> per-user profile by averaging the TF-IDF vectors of all talks that user rated **≥ 4** in the
> training split (falling back to their top-rated train talk if none qualify). That profile is then
> scored against the 200-talk candidate universe. This is a fairer test of the CB model's ranking
> quality and keeps the CB/CF comparison on equal footing.

In [62]:
# --- Content-Based (train-aware, multi-talk user profile) ---
# The CB model is unchanged (Task 5/6 TF-IDF + cosine). Only the query changed:
# average the TF-IDF rows of the talks the user rated >= 4 (fallback: top-rated train talk).
TID_TO_IDX = {int(t): i for i, t in enumerate(df_talks["talk_id"])}

_CB_PROFILE_CACHE = {}
_CB_SCORE_CACHE = {}


def _user_profile_vector(user_id, train_df, cb_tfidf, df_talks):
    """Average TF-IDF rows of talks rated >= 4 (fallback: top-rated train talk)."""
    if user_id in _CB_PROFILE_CACHE:
        return _CB_PROFILE_CACHE[user_id]

    user_rows = train_df[train_df["user_id"] == user_id]
    liked = user_rows[user_rows["rating"] >= 4]
    if liked.empty:
        liked = user_rows.sort_values("rating", ascending=False).head(1)

    idx = [TID_TO_IDX[int(t)] for t in liked["talk_id"] if int(t) in TID_TO_IDX]
    if not idx:
        _CB_PROFILE_CACHE[user_id] = None
        return None

    profile = np.asarray(cb_tfidf[idx].mean(axis=0)).ravel()
    norm = np.linalg.norm(profile)
    if norm <= 0:
        _CB_PROFILE_CACHE[user_id] = None
        return None

    profile = profile / norm
    _CB_PROFILE_CACHE[user_id] = profile
    return profile


def _user_cb_scores(user_id, train_df, cb_tfidf, df_talks):
    """Cosine scores of the whole catalogue against the user's profile (cached)."""
    if user_id in _CB_SCORE_CACHE:
        return _CB_SCORE_CACHE[user_id]
    profile = _user_profile_vector(user_id, train_df, cb_tfidf, df_talks)
    if profile is None:
        _CB_SCORE_CACHE[user_id] = None
        return None
    scores = np.asarray(cb_tfidf @ profile).ravel()
    _CB_SCORE_CACHE[user_id] = scores
    return scores


def cb_profile_scores_for_eval(user_id, train_df, exclude_ids, top_n, restrict_to_interacted=True):
    """Return {talk_id: cosine_score} for the top-n CB candidates."""
    scores = _user_cb_scores(user_id, train_df, cb_tfidf, df_talks)
    if scores is None:
        return {}

    ranked = scores.argsort()[::-1]
    interacted_tids = set(interactions["talk_id"].unique())

    rec = {}
    for i in ranked:
        tid = int(df_talks.loc[i, "talk_id"])
        if restrict_to_interacted and tid not in interacted_tids:
            continue
        if tid in exclude_ids:
            continue
        rec[tid] = float(scores[i])
        if len(rec) >= top_n:
            break
    return rec


def cb_profile_recommend_for_eval(user_id, train_df, exclude_ids, top_n, restrict_to_interacted=True):
    """List wrapper used by the evaluation loop."""
    return list(cb_profile_scores_for_eval(user_id, train_df, exclude_ids, top_n, restrict_to_interacted).keys())

In [63]:
# --- Collaborative Filtering (train-aware, mean-centered) ---
def cf_scores_for_eval(user_id, exclude_ids, top_n, k_neighbors=None):
    """Return {talk_id: (pred_rating, n_raters)} sorted by score then n_raters."""
    if k_neighbors is None:
        k_neighbors = K_NEIGHBORS
    if user_id not in train_user_item.index:
        return {}
    user_r = train_user_item.loc[user_id]
    if user_r.dropna().empty:
        return {}

    sims = train_sim_df.loc[user_id].drop(user_id, errors="ignore")
    nbrs = sims.sort_values(ascending=False).head(k_neighbors)
    nbr_ids, nbr_w = nbrs.index, nbrs.values

    nbr_centered = train_centered.loc[nbr_ids]
    cand_mask = user_r.isna()
    nbr_has_data = train_user_item.loc[nbr_ids].notna()
    target_mean = float(train_user_means[user_id])

    preds = {}
    for tid in train_user_item.columns:
        if not cand_mask[tid]:
            continue
        rated = nbr_has_data[tid]
        if not rated.any():
            continue
        w = nbr_w[rated.values]
        centered_r = nbr_centered.loc[rated, tid].values
        denom = np.sum(np.abs(w))
        if denom <= 0:
            continue
        pred_dev = np.dot(w, centered_r) / denom
        preds[tid] = (float(np.clip(target_mean + pred_dev, 1.0, 5.0)), int(rated.sum()))

    ranked = sorted(preds.items(), key=lambda x: (-x[1][0], -x[1][1]))
    rec = {}
    for tid, (score, n) in ranked:
        if tid in exclude_ids:
            continue
        rec[tid] = (score, n)
        if len(rec) >= top_n:
            break
    return rec


def cf_recommend_for_eval(user_id, exclude_ids, top_n, k_neighbors=None):
    """List wrapper used by the evaluation loop."""
    return list(cf_scores_for_eval(user_id, exclude_ids, top_n, k_neighbors).keys())

In [64]:
# --- Hybrid (min-max on RAW scores, α-weighted) ---
def _min_max(d):
    if not d:
        return {}
    vals = list(d.values())
    lo, hi = min(vals), max(vals)
    if hi - lo < 1e-12:
        return {k: 1.0 for k in d}
    return {k: (v - lo) / (hi - lo) for k, v in d.items()}

In [65]:
def hybrid_recommend_for_eval(user_id, exclude_ids, top_n, alpha=0.5, pool=30):
    """
    Hybrid using min-max normalization of raw CB and CF scores, then
    alpha * CB_norm + (1 - alpha) * CF_norm * CF_conf.

    CF_conf = min(n_raters / 5, 1.0); talks outside the CF universe get 0.
    Already-rated talks are filtered BEFORE normalization so the [0, 1]
    range is computed only over valid candidates.

    alpha = 0.0 returns pure CF and alpha = 1.0 returns pure CB, so the
    endpoints of the sweep are exactly the single-model results.
    """
    if alpha <= 0.0:
        return cf_recommend_for_eval(user_id, exclude_ids, top_n)
    if alpha >= 1.0:
        return cb_profile_recommend_for_eval(user_id, train_df, exclude_ids, top_n, restrict_to_interacted=True)

    cb = cb_profile_scores_for_eval(user_id, train_df, exclude_ids, pool, restrict_to_interacted=True)
    cf = cf_scores_for_eval(user_id, exclude_ids, pool)

    if not cb and not cf:
        return []

    cb_norm = _min_max(cb)
    cf_raw = {tid: sc for tid, (sc, n) in cf.items()}
    cf_norm = _min_max(cf_raw)
    cf_conf = {tid: min(n / 5.0, 1.0) for tid, (sc, n) in cf.items()}

    merged = {}
    for tid in cb:
        merged[tid] = alpha * cb_norm[tid] + (1 - alpha) * cf_norm.get(tid, 0.0) * cf_conf.get(tid, 0.0)
    for tid in cf:
        if tid not in merged:
            merged[tid] = (1 - alpha) * cf_norm[tid] * cf_conf[tid]

    ranked = sorted(merged.items(), key=lambda x: -x[1])
    return [tid for tid, _ in ranked][:top_n]

##### Evaluation Loop

For each evaluable user:
1. Build the user's CB profile from training talks rated **≥ 4** (fallback: top-rated train talk).
2. Generate top-K recommendations from each model.
3. A hit = recommended `talk_id` is in the user's **relevant held-out set**.
4. Accumulate Precision@K, Recall@K, and NDCG@K across users.

In [66]:
def precision_at_k(recommended, relevant):
    if not recommended:
        return 0.0
    return len(set(recommended) & relevant) / len(recommended)


def recall_at_k(recommended, relevant):
    if not relevant:
        return 0.0
    return len(set(recommended) & relevant) / len(relevant)


def ndcg_at_k(recommended, relevant, k):
    """Binary-relevance NDCG@k."""
    if not recommended or not relevant:
        return 0.0
    dcg = 0.0
    for rank, tid in enumerate(recommended[:k], start=1):
        if tid in relevant:
            dcg += 1.0 / np.log2(rank + 1)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(r + 1) for r in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_models(users, k_values=(5, 10), alpha=0.5):
    results = {m: {} for m in ["CB", "CF", "Hybrid"]}
    for m in results:
        for k in k_values:
            results[m][f"P@{k}"] = []
            results[m][f"R@{k}"] = []
            results[m][f"NDCG@{k}"] = []

    for uid in users:
        # Guard: a user with no interaction left in train cannot be scored.
        if uid not in train_user_item.index:
            continue

        relevant = relevant_heldout[uid]

        # User's rated talks in TRAIN -> exclude from recs
        user_train = train_user_item.loc[uid].dropna()
        if user_train.empty:
            continue
        rated_train = set(user_train.index)

        for k in k_values:
            cb_recs = cb_profile_recommend_for_eval(uid, train_df, rated_train, k, restrict_to_interacted=True)
            cf_recs = cf_recommend_for_eval(uid, rated_train, k)
            hy_recs = hybrid_recommend_for_eval(uid, rated_train, k, alpha=alpha)

            results["CB"][f"P@{k}"].append(precision_at_k(cb_recs, relevant))
            results["CB"][f"R@{k}"].append(recall_at_k(cb_recs, relevant))
            results["CB"][f"NDCG@{k}"].append(ndcg_at_k(cb_recs, relevant, k))

            results["CF"][f"P@{k}"].append(precision_at_k(cf_recs, relevant))
            results["CF"][f"R@{k}"].append(recall_at_k(cf_recs, relevant))
            results["CF"][f"NDCG@{k}"].append(ndcg_at_k(cf_recs, relevant, k))

            results["Hybrid"][f"P@{k}"].append(precision_at_k(hy_recs, relevant))
            results["Hybrid"][f"R@{k}"].append(recall_at_k(hy_recs, relevant))
            results["Hybrid"][f"NDCG@{k}"].append(ndcg_at_k(hy_recs, relevant, k))

    rows = []
    for model in ["CB", "CF", "Hybrid"]:
        for metric, vals in results[model].items():
            rows.append({"Model": model, "Metric": metric, "Score": round(float(np.mean(vals)), 4)})
    return pd.DataFrame(rows)


metrics_df = evaluate_models(evaluable_users, k_values=(5, 10), alpha=0.5)
metrics_df

,Model,Metric,Score
0,CB,P@5,0.0397
1,CB,R@5,0.0581
2,CB,NDCG@5,0.0476
3,CB,P@10,0.0352
4,CB,R@10,0.0986
5,CB,NDCG@10,0.0660
6,CF,P@5,0.0196
7,CF,R@5,0.0309
8,CF,NDCG@5,0.0285
9,CF,P@10,0.0196


In [67]:
# Wide table for readability
pivot = metrics_df.pivot(index="Model", columns="Metric", values="Score")
pivot = pivot[["P@5", "R@5", "NDCG@5", "P@10", "R@10", "NDCG@10"]]
pivot

Metric,P@5,R@5,NDCG@5,P@10,R@10,NDCG@10
Model,,,,,,
CB,0.0397,0.0581,0.0476,0.0352,0.0986,0.0660
CF,0.0196,0.0309,0.0285,0.0196,0.0650,0.0426
Hybrid,0.0380,0.0588,0.0473,0.0335,0.0957,0.0642


##### Hybrid α Sweep

Evaluate the hybrid at α ∈ `{0.0, 0.3, 0.5, 0.7, 1.0}` to find the best weighting. α = 0.0 is pure CF;
α = 1.0 is pure CB.

In [68]:
alpha_rows = []
for a in [0.0, 0.3, 0.5, 0.7, 1.0]:
    df_a = evaluate_models(evaluable_users, k_values=(5, 10), alpha=a)
    row = {"alpha": a}
    for _, r in df_a.iterrows():
        if r["Model"] == "Hybrid":
            row[r["Metric"]] = r["Score"]
    alpha_rows.append(row)

alpha_df = pd.DataFrame(alpha_rows)
alpha_df = alpha_df[["alpha", "P@5", "R@5", "NDCG@5", "P@10", "R@10", "NDCG@10"]]
alpha_df

,alpha,P@5,R@5,NDCG@5,P@10,R@10,NDCG@10
0,0.0,0.0196,0.0309,0.0285,0.0196,0.0650,0.0426
1,0.3,0.0352,0.0530,0.0439,0.0327,0.0990,0.0635
2,0.5,0.0380,0.0588,0.0473,0.0335,0.0957,0.0642
3,0.7,0.0372,0.0547,0.0455,0.0354,0.1006,0.0664
4,1.0,0.0397,0.0581,0.0476,0.0352,0.0986,0.0660


In [69]:
# Best alpha by P@5 (primary metric)
best_row = alpha_df.loc[alpha_df["P@5"].idxmax()]
print(f"Best α by P@5: {best_row['alpha']}  -> P@5={best_row['P@5']}, R@5={best_row['R@5']}, NDCG@5={best_row['NDCG@5']}")
best_alpha = float(best_row["alpha"])
print(f"Selected final α = {best_alpha}")

Best α by P@5: 1.0  -> P@5=0.0397, R@5=0.0581, NDCG@5=0.0476
Selected final α = 1.0


##### Model Comparison Table

Final comparison at the selected α, showing **Precision@5, Recall@5, NDCG@5** for:
- **CB** at α = 1.0 (pure content-based profile),
- **CF** at α = 0.0 (pure mean-centered CF),
- **Hybrid** at its best α.

In [70]:
final_metrics = evaluate_models(evaluable_users, k_values=(5, 10), alpha=best_alpha)
final_pivot = final_metrics.pivot(index="Model", columns="Metric", values="Score")

comparison_table = pd.DataFrame({
    "P@5": [final_pivot.loc["CB", "P@5"], final_pivot.loc["CF", "P@5"], final_pivot.loc["Hybrid", "P@5"]],
    "R@5": [final_pivot.loc["CB", "R@5"], final_pivot.loc["CF", "R@5"], final_pivot.loc["Hybrid", "R@5"]],
    "NDCG@5": [final_pivot.loc["CB", "NDCG@5"], final_pivot.loc["CF", "NDCG@5"], final_pivot.loc["Hybrid", "NDCG@5"]],
}, index=["CB (α=1.0)", "CF (α=0.0)", f"Hybrid (α={best_alpha})"])
comparison_table["Notes"] = [
    "Averaged TF-IDF profile of talks rated ≥4; 200-talk universe",
    "Mean-centered user-based CF; 200-talk universe",
    f"min-max CB + confidence-weighted CF (α={best_alpha})",
]
comparison_table

,P@5,R@5,NDCG@5,Notes
CB (α=1.0),0.0397,0.0581,0.0476,Averaged TF-IDF profile of talks rated ≥4; 200-talk universe
CF (α=0.0),0.0196,0.0309,0.0285,Mean-centered user-based CF; 200-talk universe
Hybrid (α=1.0),0.0397,0.0581,0.0476,min-max CB + confidence-weighted CF (α=1.0)


##### Interpretations

**Reading the numbers:**

- **P@5** = fraction of the 5 recommended talks the user actually liked (rated ≥ 3).
- **R@5** = fraction of the user's held-out relevant talks recovered in the top 5.
- **NDCG@5** = rank-aware precision: hits near the top count more.
- **P@10 / R@10 / NDCG@10** = same, for the top 10. Recall rises with K because there are more slots.

**Results on the 500-user dataset (489 evaluable users):**

1. **CB is the strongest model** on the matched candidate universe: P@5 = **0.0397**, R@5 = **0.0581**, NDCG@5 = **0.0476** (P@10 = 0.0352, R@10 = 0.0986, NDCG@10 = 0.0660).
2. **CF is weaker**: P@5 = **0.0196**, R@5 = **0.0309**, NDCG@5 = **0.0285**. Mean-centering improved it over the raw zero-fill version, but at ~90% sparsity its neighbourhood averages remain noisy.
3. **The hybrid at α = 0.5 nearly matches CB** (P@5 = **0.0380**, NDCG@5 = **0.0473**) and has a marginally higher R@5 (**0.0588** vs 0.0581). It does not beat CB on the primary metric.
4. **The α sweep peaks at α = 1.0**: P@5 goes **0.0196** (α = 0.0) -> **0.0352** (0.3) -> **0.0380** (0.5) -> **0.0372** (0.7) -> **0.0397** (1.0). The best α is **1.0**, where the hybrid is exactly CB. CF does not add measurable value on this dataset.
5. **The candidate universe matters.** The appendix shows CB scoring P@5 = **0.0020** when it may rank all 4,005 talks while the holdout lives in the 200-talk pool - about **20x lower** than the matched result. That gap is an evaluation artefact, not a model-quality difference.

**Context:** random top-5 from the 200-talk pool would score roughly P@5 ≈ 0.017 at this relevance density (~3.3 relevant items per user). CB scores about **2.3x** that; CF is only slightly above random.

**Bottom line:** after removing the candidate-universe confound and using a multi-talk CB profile, the data still selects **CB**; the hybrid's best setting is α = 1.0 (pure CB). This is reported as-is - no α or threshold was tuned to force a hybrid win.

##### Strengths and Weaknesses of Each Model

| Model | Strengths | Weaknesses |
|---|---|---|
| **Content-Based** | Works for any seed/profile (no item cold-start). Explainable - recommendations trace to the user's liked talks. Multi-talk profile improves ranking over a single seed. Strongest P@5 and NDCG@5 on the matched 200-talk universe. | Ignores other users; recommends near-duplicates of the profile talks. Needs a user history (or seed). Low diversity. |
| **Collaborative Filtering** | Directly models user preference. Mean-centering removes per-user scale bias. Personalised - different users get different recs. Produces a list for every user at this density. | Cold-start for new users and items. Signed/noisy similarity at 90% sparsity. Popularity bias. Not explainable. |
| **Hybrid** | Balances topical relevance and personalisation. Confidence weighting suppresses thin-evidence CF. Nearly matches CB at α = 0.5 and adds a small recall gain. | At this density it does not beat CB on P@5/NDCG@5; the best α collapses to CB (α = 1.0). Requires tuning α. Adds latency (two models). |

---

##### Limitations Observed

1. **Sparsity.** 90.2% of the user-item matrix is missing (9,778 of 100,000 cells filled). Mean-centered user-user similarity is near zero on average (max ~0.47), so CF predictions are noisy.
2. **Holdout size.** 30% of ~20 ratings per user = ~6 held-out items, of which ~3.3 are relevant. Precision/recall at this sample size is noisy.
3. **Cold-start.** A brand-new user has no CF history and no liked talks to build a CB profile. All 500 simulated users have ≥7 ratings, so neither case occurs here.
4. **Relevance threshold.** Relevance is `rating ≥ 3` (58% of interactions). Changing the threshold would change every metric; it is held fixed for comparability.
5. **Profile threshold.** CB averages talks rated **≥ 4** (falling back to the top-rated train talk). A different threshold, or a different fallback, would change the CB list.
6. **No negative feedback.** Low ratings (1–2) come from low tag overlap rather than explicit dislike, so precision is biased toward not-recommending wrongly.

##### Final Model Configuration

**Selected model:** **Content-Based (CB)** with a multi-talk user profile.

**Why:**

1. **Best P@5 and NDCG@5 on held-out data** - P@5 = 0.0397 and NDCG@5 = 0.0476, versus 0.0196 / 0.0285 for CF and 0.0380 / 0.0473 for the hybrid at α = 0.5.
2. **The hybrid's best α is 1.0**, where it reduces to pure CB. No intermediate α beats CB on P@5, so the CF component adds no measurable value on this dataset.
3. **CB is simpler and more explainable** - a single model, no α dial, and every recommendation can be traced to the user's liked talks.

**Trade-offs accepted:**
- CB ignores other users' behaviour, so it cannot exploit collaborative signals.
- CB recommends talks topically close to the user's liked talks; diversity is limited.
- The profile needs at least one rated talk; fall back to the highest-rated train talk when none are ≥ 4 (already done here).

**Not selected:**
- **CF alone** - lower P@5 (0.0196) and NDCG@5 (0.0285) than CB at this density.
- **Hybrid at α = 0.3–0.7** - P@5 between 0.0352 and 0.0380, below CB.
- **Hybrid at α = 1** - identical to CB; carrying the CF branch adds no benefit.

**Note on recall:** at α = 0.5 the hybrid's R@5 (0.0588) is marginally above CB's (0.0581), but its P@5 and NDCG@5 are lower. Since the primary metric is P@5, CB is selected; the recall difference is within noise at this holdout size.

**Next steps (Task 11 - Streamlit deployment):**
1. Build `app.py` with:
   - User selector (dropdown of `user_id`s)
   - Seed talk input (autocomplete from `df_talks["title"]`)
   - α slider (0–1), defaulting to the evaluated best α
   - Top-N selector
2. Display a side-by-side comparison: CB, CF, Hybrid.
3. Cache all artefacts with `@st.cache_resource` for fast page loads.
4. Document the cold-start flow (new user -> CB-only with a clear notice).

In [71]:
# Final summary print
winner = comparison_table["P@5"].idxmax()

print("="*80)
print("FINAL MODEL CONFIGURATION")
print("="*80)
print(f"Selected model:  {winner}")
print(f"Best alpha:      {best_alpha}")
print(f"Evaluable users: {len(evaluable_users)} / {N_USERS}")
print(f"Holdout:         30% per user")
print(f"Relevance:       rating >= {RELEVANCE_THRESHOLD}")
print(f"K values:        5 and 10")
print()
print("Final metrics:")
print(comparison_table[["P@5", "R@5", "NDCG@5"]].to_string())
if best_alpha == 1.0:
    print("\nNote: the hybrid's best α is 1.0 - at that setting it is identical to pure CB.")
elif best_alpha == 0.0:
    print("\nNote: the hybrid's best α is 0.0 - at that setting it is identical to pure CF.")

FINAL MODEL CONFIGURATION
Selected model:  CB (α=1.0)
Best alpha:      1.0
Evaluable users: 489 / 500
Holdout:         30% per user
Relevance:       rating >= 3
K values:        5 and 10

Final metrics:
                   P@5     R@5  NDCG@5
CB (α=1.0)      0.0397  0.0581  0.0476
CF (α=0.0)      0.0196  0.0309  0.0285
Hybrid (α=1.0)  0.0397  0.0581  0.0476

Note: the hybrid's best α is 1.0 - at that setting it is identical to pure CB.


##### Appendix: CB Without Candidate-Universe Restriction (Unfair Comparison)

The main comparison restricts CB to the same **200-talk universe** as CF. For reference, this appendix
runs the same multi-talk CB profile with `restrict_to_interacted=False`, so it may recommend from all
**4,005** talks in the catalogue while the held-out items still come from the 200 interacted talks.

This is the old-style, unfair setup: CB's candidate pool is 20x larger than the evaluation pool, so its
chance of hitting a held-out talk drops sharply. The result is shown **for comparison only and is not
used for model selection**.

In [72]:
# ---------------------------------------------------------------------------
# APPENDIX: CB without candidate-universe restriction (unfair comparison).
# Same multi-talk profile; only the candidate pool changes.
# For comparison only - not used for model selection.
# ---------------------------------------------------------------------------
def evaluate_cb_unrestricted(users, k_values=(5, 10)):
    scores = {}
    for k in k_values:
        scores[f"P@{k}"] = []
        scores[f"R@{k}"] = []
        scores[f"NDCG@{k}"] = []
    for uid in users:
        if uid not in train_user_item.index:
            continue
        user_train = train_user_item.loc[uid].dropna()
        if user_train.empty:
            continue
        rated_train = set(user_train.index)
        for k in k_values:
            recs = cb_profile_recommend_for_eval(uid, train_df, rated_train, k, restrict_to_interacted=False)
            scores[f"P@{k}"].append(precision_at_k(recs, relevant_heldout[uid]))
            scores[f"R@{k}"].append(recall_at_k(recs, relevant_heldout[uid]))
            scores[f"NDCG@{k}"].append(ndcg_at_k(recs, relevant_heldout[uid], k))
    return {m: round(float(np.mean(v)), 4) for m, v in scores.items()}


unfair_cb = evaluate_cb_unrestricted(evaluable_users)
matched_cb = final_pivot.loc["CB", ["P@5", "R@5", "NDCG@5", "P@10", "R@10", "NDCG@10"]].to_dict()

appendix_table = pd.DataFrame({
    "CB (matched 200-talk universe)": matched_cb,
    "CB (full 4,005-talk catalogue)": unfair_cb,
}).T
appendix_table

,P@5,R@5,NDCG@5,P@10,R@10,NDCG@10
CB (matched 200-talk universe),0.0397,0.0581,0.0476,0.0352,0.0986,0.0660
"CB (full 4,005-talk catalogue)",0.0020,0.0019,0.0025,0.0020,0.0046,0.0037


#### 📄 Task Report

The written report for this notebook lives at [`reports/task_10_observations.md`](../reports/task_10_observations.md).

- Markdown: [`reports/task_10_observations.md`](../reports/task_10_observations.md)
- PDF: [`reports/pdf/task_10_observations.pdf`](../reports/pdf/task_10_observations.pdf)

If a link does not open inside your notebook viewer, open the file directly from the project root.
